#Schritt 4: Ansatz 2:Transformer

Auf GoEmotions fine-getuntes Modell als zweiten Ansatz (Demszky et al., 2020). Das Modell wird von HuggingFace heruntergeladen.

In [ ]:
import pandas as pd
from transformers import pipeline
MODELL_NAME = 'SamLowe/roberta-base-go_emotions'

In [ ]:
def lade_emotions_modell():
    emotion_klassifikator = pipeline(task='text-classification', model=MODELL_NAME, top_k=None)
    return emotion_klassifikator

In [ ]:
def erkenne_emotionen(text, klassifikator, max_zeichen=512):
    gekürzter_text = text[:max_zeichen]
    ergebnis = klassifikator(gekürzter_text)
    emotionen = ergebnis[0]
    return emotionen

In [ ]:
def hole_top_emotion(emotionen):
    top = emotionen[0]
    for emotion in emotionen:
        if emotion['score'] > top['score']:
            top = emotion
    return (top['label'], top['score'])

In [ ]:
def berechne_emotionen_für_datensatz(daten):
    klassifikator = lade_emotions_modell()
    top_emotion_liste = []
    top_score_liste = []
    alle_emotionen_liste = []
    for index, zeile in daten.iterrows():
        text = zeile['lyrics_clean']
        emotionen = erkenne_emotionen(text, klassifikator)
        top_label, top_score = hole_top_emotion(emotionen)
        top_emotion_liste.append(top_label)
        top_score_liste.append(top_score)
        alle_emotionen_liste.append(emotionen)
        titel = zeile['title']
        print(f'{titel}: stärkste Emotion = {top_label} ({top_score:.2f})')
    daten['top_emotion'] = top_emotion_liste
    daten['top_emotion_score'] = top_score_liste
    daten['alle_emotionen'] = alle_emotionen_liste
    return daten

## Pipeline ausführen

In [ ]:
daten = pd.read_csv('data/daten_mit_vader.csv')
daten = berechne_emotionen_für_datensatz(daten)
daten.to_csv('data/daten_mit_emotionen.csv', index=False)
print('Ergebnisse gespeichert unter: data/daten_mit_emotionen.csv')